# Estimativas e tamanhos de efeito

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_05/01_estimativas_e_tamanhos_de_efeito.ipynb)

O guia definiu uma comparação como uma cadeia argumentativa. Este notebook
começa pela pergunta mais concreta: **quanto os grupos diferem na amostra ou no
corpus observado?**

## 1. Unidades, grupos e quantidade de interesse

Compare apenas unidades que tenham significado compatível. Aqui cada linha é um
documento; usaremos `local` para formar grupos e `palavras` como extensão
simulada. A média responde sobre a extensão média; a mediana, sobre a posição
central resistente a extremos; a proporção, sobre a frequência de uma condição.

![Diferença observada, incerteza do procedimento e relevância substantiva aparecem como três perguntas distintas ligadas em sequência.](imagens/01_diferenca_incerteza_relevancia.svg)

Defina a quantidade antes de calcular. Se $\hat{\theta}_A$ e
$\hat{\theta}_B$ são estimativas nos grupos, a diferença absoluta é:

$$
\Delta=\hat{\theta}_A-\hat{\theta}_B.
$$

O sinal depende da ordem declarada.

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_05'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = []
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

In [ ]:
import pandas as pd
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd())) if str(Path.cwd()) not in sys.path else None
from graficos import estimativas_grupos
from IPython.display import display

dados = pd.read_csv("dados/documentos.csv")
grupos = dados.groupby("local")["palavras"]
resumo_grupos = grupos.agg(["count", "mean", "median", "std"])
display(resumo_grupos.round(2))
valores_capital = dados.loc[dados["local"].eq("Capital"), "palavras"].tolist()
valores_interior = dados.loc[dados["local"].eq("Interior"), "palavras"].tolist()
estimativas_grupos(valores_capital, valores_interior, "Capital", "Interior")

A tabela e o gráfico mostram distribuição, sobreposição e um caso extremo. Por
isso, uma única medida não deve apagar a forma dos grupos.

## 2. Médias, medianas e proporções

Para cada medida, calcularemos Capital menos Interior. A proporção corresponde
aos documentos cujo tema dominante é `educação`; ela não mede quanto cada texto
discute educação.

In [ ]:
por_local = dados.groupby("local")
medidas = pd.DataFrame({
    "media_palavras": por_local["palavras"].mean(),
    "mediana_palavras": por_local["palavras"].median(),
    "proporcao_educacao": por_local["tema"].apply(lambda s: s.eq("educação").mean()),
})
diferencas = medidas.loc["Capital"] - medidas.loc["Interior"]
display(medidas.round(3))
diferencas.rename("Capital menos Interior").round(3)

Média e mediana respondem a aspectos diferentes, e a diferença de proporções é
lida em pontos percentuais. A escolha deve preceder a observação do resultado.

## 3. Diferenças absolutas e relativas

Tomando B como referência, a diferença relativa é:

$$
\Delta_{rel}=\frac{\hat{\theta}_A-\hat{\theta}_B}{\hat{\theta}_B}.
$$

Ela muda ao inverter a referência e se torna instável quando o denominador se
aproxima de zero. Sempre apresente também a diferença absoluta e a unidade.

In [ ]:
media_capital = medidas.loc["Capital", "media_palavras"]
media_interior = medidas.loc["Interior", "media_palavras"]
diferenca_absoluta = media_capital - media_interior
diferenca_relativa = diferenca_absoluta / media_interior
pd.Series({
    "diferença absoluta (palavras)": diferenca_absoluta,
    "diferença relativa a Interior": diferenca_relativa,
}).round(3)

A diferença relativa produz uma narrativa proporcional, mas não informa a
variabilidade interna. Para comparar escalas, pode-se padronizar a diferença;
ainda assim, a escala original permanece indispensável.

## 4. Tamanho de efeito

Para duas médias, uma versão introdutória da diferença padronizada é:

$$
d=\frac{\bar{x}_A-\bar{x}_B}{s_p},\qquad
s_p=\sqrt{\frac{(n_A-1)s_A^2+(n_B-1)s_B^2}{n_A+n_B-2}}.
$$

`d` expressa a diferença em desvios-padrão combinados. Rótulos universais como
“pequeno” ou “grande” não substituem conhecimento substantivo, desenho dos dados
ou diferença bruta.

In [ ]:
import math

a = dados.loc[dados["local"].eq("Capital"), "palavras"]
b = dados.loc[dados["local"].eq("Interior"), "palavras"]
desvio_combinado = math.sqrt(
    ((len(a)-1)*a.var(ddof=1) + (len(b)-1)*b.var(ddof=1))
    / (len(a)+len(b)-2)
)
d_padronizado = (a.mean() - b.mean()) / desvio_combinado
pd.Series({"diferença bruta": a.mean()-b.mean(), "d padronizado": d_padronizado}).round(3)

O efeito padronizado resume separação relativa à dispersão, mas pode mudar com
um único caso influente. A etapa seguinte retorna aos registros.

## 5. Sensibilidade e relevância substantiva

Compare a diferença de médias com e sem D023, o extremo deliberado. Depois leia
os textos dos documentos com maiores e menores valores. Pergunte se `palavras`
representa extensão documental, intensidade do tema ou apenas um campo simulado.

In [ ]:
sem_extremo = dados.loc[~dados["id_documento"].eq("D023")]
sensibilidade = pd.Series({
    "todos os registros": diferenca_absoluta,
    "sem D023": (
        sem_extremo.loc[sem_extremo["local"].eq("Capital"), "palavras"].mean()
        - sem_extremo.loc[sem_extremo["local"].eq("Interior"), "palavras"].mean()
    ),
})
display(sensibilidade.round(2))
dados.nlargest(3, "palavras")[["id_documento", "local", "palavras", "texto"]]

## Atividade integrada — quadro de estimativas

**Modalidade:** individual e revisão em dupla. **Tempo:** 25 + 10 minutos.

1. escolha dois grupos e uma quantidade coerente com sua pergunta;
2. apresente a distribuição e ao menos duas medidas pertinentes;
3. calcule diferença absoluta e, se defensável, relativa ou padronizada;
4. inspecione dois casos que qualifiquem o resultado;
5. escreva o que a diferença significa e o que não significa.

**Minha estimativa e justificativa:** Escreva aqui.

**Casos inspecionados:** Escreva aqui.

**Relevância substantiva e limite:** Escreva aqui.

Leve o quadro ao Notebook 02. Ele será acompanhado por uma avaliação explícita
da variabilidade e das condições em que a inferência é pertinente.